# PHASE 11 - Save the Best Model

The ablation study selected the time + weather + lag feature set using validation RMSE. This notebook trains that selected configuration on train plus validation data and saves it for later prediction.

The test set remains untouched until the final evaluation in this notebook.

In [1]:
from pathlib import Path
from time import perf_counter
import pickle

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

def find_project_root():
    for candidate in [Path("."), Path("..")]:
        if (candidate / "data/processed/train.csv").exists():
            return candidate
    raise FileNotFoundError("Run PHASE 3 first.")

project_root = find_project_root()
processed_dir = project_root / "data/processed"
metrics_dir = project_root / "results/metrics"
models_dir = project_root / "models"
train_df = pd.read_csv(processed_dir / "train.csv", parse_dates=["timestamp", "dteday"])
validation_df = pd.read_csv(processed_dir / "validation.csv", parse_dates=["timestamp", "dteday"])
test_df = pd.read_csv(processed_dir / "test.csv", parse_dates=["timestamp", "dteday"])
ablation_df = pd.read_csv(metrics_dir / "ablation_study_metrics.csv")
search_df = pd.read_csv(metrics_dir / "xgboost_validation_search.csv")

best_validation = ablation_df[ablation_df["split"] == "validation"].sort_values("RMSE").iloc[0]
assert best_validation["experiment"] == "C_time_weather_lag"
time_features = ["yr", "mnth", "hr", "holiday", "weekday", "workingday", "season", "hour", "day", "month", "year", "day_of_week", "day_of_year", "is_weekend", "is_workingday", "rush_hour"]
weather_features = ["weathersit", "temp", "atemp", "hum", "windspeed"]
lag_features = ["lag_1", "lag_2", "lag_24", "lag_168"]
feature_columns = time_features + weather_features + lag_features
target_column = "cnt"
train_validation_df = pd.concat([train_df, validation_df], ignore_index=True)

print(f"Selected feature set: {best_validation['experiment']}")
print(f"Feature count: {len(feature_columns)}")
print(f"Final training rows: {len(train_validation_df)}")

Selected feature set: C_time_weather_lag
Feature count: 25
Final training rows: 14629


In [2]:
best_xgb = search_df.sort_values("RMSE").iloc[0]
xgb_parameters = {
    "n_estimators": int(best_xgb["n_estimators"]),
    "max_depth": int(best_xgb["max_depth"]),
    "learning_rate": float(best_xgb["learning_rate"]),
    "min_child_weight": int(best_xgb["min_child_weight"]),
    "objective": "reg:squarederror",
    "tree_method": "hist",
    "device": "cpu",
    "n_jobs": -1,
    "random_state": 42,
}
print("Final XGBoost parameters:", xgb_parameters)

Final XGBoost parameters: {'n_estimators': 400, 'max_depth': 6, 'learning_rate': 0.03, 'min_child_weight': 3, 'objective': 'reg:squarederror', 'tree_method': 'hist', 'device': 'cpu', 'n_jobs': -1, 'random_state': 42}


## Fit final model on train + validation

In [3]:
final_model = XGBRegressor(**xgb_parameters)
start_time = perf_counter()
final_model.fit(train_validation_df[feature_columns], train_validation_df[target_column])
training_time = perf_counter() - start_time
test_predictions = final_model.predict(test_df[feature_columns])

test_metrics = {
    "model": "XGBoost final",
    "feature_set": best_validation["experiment"],
    "split": "test",
    "MAE": mean_absolute_error(test_df[target_column], test_predictions),
    "RMSE": mean_squared_error(test_df[target_column], test_predictions) ** 0.5,
    "R2": r2_score(test_df[target_column], test_predictions),
    "Training Time": training_time,
}
final_metrics = pd.DataFrame([test_metrics])
display(final_metrics.round(4))
final_metrics.to_csv(metrics_dir / "best_model_final_metrics.csv", index=False)
print(f"Training time: {training_time:.2f} seconds")

,model,feature_set,split,MAE,RMSE,R2,Training Time
0,XGBoost final,C_time_weather_lag,test,29.146,47.7517,0.9502,0.5351


Training time: 0.54 seconds


In [4]:
model_bundle = {
    "model": final_model,
    "feature_columns": feature_columns,
    "target_column": target_column,
    "feature_set": best_validation["experiment"],
    "xgb_parameters": xgb_parameters,
}
model_path = models_dir / "best_model_xgboost.pkl"
with model_path.open("wb") as model_file:
    pickle.dump(model_bundle, model_file)

importance = pd.DataFrame({"feature": feature_columns, "importance": final_model.feature_importances_}).sort_values("importance", ascending=False)
importance.to_csv(metrics_dir / "best_model_feature_importance.csv", index=False)
print(f"Saved best model to: {model_path}")

Saved best model to: ..\models\best_model_xgboost.pkl


## Phase 11 conclusion

The selected XGBoost model is saved with its feature schema and parameters. PHASE 12 can load this bundle for the Streamlit prediction interface.